# Session V: File I/O and Structured Data

Persist data using text files, JSON, and CSV, and learn how to design file-based mini databases.

## Exercise 1: Daily Journal App
**Objective:** Practice reading, writing, and appending to text files.

Build a simple daily journal application that stores entries in a plain text file (`journal.txt`). Each entry should include a timestamp and the user's text.

**Tasks:**
1. `add_entry(filename, text)` → Append a new journal entry with the current date/time to the file.
2. `read_journal(filename)` → Read and display all entries from the file. Handle the case where the file doesn't exist yet.
3. `search_journal(filename, keyword)` → Read the file and return only lines that contain the keyword (case-insensitive).
4. `count_entries(filename)` → Return the total number of entries in the journal.
5. `run_journal()` → A menu-driven interface: [A]dd entry, [R]ead all, [S]earch, [C]ount, [Q]uit.

In [33]:
from datetime import datetime

JOURNAL_FILE = "journal.txt"

def add_entry(filename, text):
    """Append a timestamped journal entry to the file."""
    # TODO: Open file in append mode
    # TODO: Write the current datetime + the entry text
    with open(filename, 'a') as f:
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        f.write(f"{timestamp} - {text}\n")
    

def read_journal(filename):
    """Read and print all entries. Handle FileNotFoundError."""
    # TODO: Try to open and read the file
    # TODO: Print each line (stripped)
    # TODO: Handle missing file gracefully
    try:
        with open(filename, 'r') as f:
            for line in f:
                print(line.strip())
    except FileNotFoundError:
        print(f"No journal entries found. The file '{filename}' does not exist.")

def search_journal(filename, keyword):
    """Return a list of lines containing the keyword (case-insensitive)."""
    # TODO: Read all lines
    # TODO: Filter lines that contain the keyword
    # TODO: Handle missing file
    try:
        with open(filename, 'r') as f:
            return [line.strip() for line in f if keyword.lower() in line.lower()]
    except FileNotFoundError:
        print(f"File '{filename}' not found.")
        return []

def count_entries(filename):
    """Count and return the number of journal entries."""
    # TODO: Read lines and count non-empty entries
    # TODO: Handle missing file (return 0)
    try:
        with open(filename, 'r') as f:
            return len([line.strip() for line in f if line.strip()])
    except FileNotFoundError:
        print(f"File '{filename}' not found.")
        return 0
    

def run_journal():
    """Menu-driven journal interface."""
    # TODO: while True loop with menu [A]dd [R]ead [S]earch [C]ount [Q]uit
    # TODO: Route to the correct function based on user input
    while True:
        print("\nJournal Menu:")
        print("[A]dd entry")
        print("[R]ead entries")
        print("[S]earch entries")
        print("[C]ount entries")
        print("[Q]uit")
        
        choice = input("Choose an option: ").strip().upper()
        
        if choice == 'A':
            text = input("Enter your journal entry: ")
            add_entry(JOURNAL_FILE, text)
            print("Entry added.")
        elif choice == 'R':
            read_journal(JOURNAL_FILE)
        elif choice == 'S':
            keyword = input("Enter keyword to search: ")
            results = search_journal(JOURNAL_FILE, keyword)
            if results:
                print("Search results:")
                for line in results:
                    print(line)
            else:
                print("No entries found with that keyword.")
        elif choice == 'C':
            total = count_entries(JOURNAL_FILE)
            print(f"Total entries: {total}")
        elif choice == 'Q':
            print("Exiting the journal. Goodbye!")
            break
        else:
            print("Invalid option. Please try again.")


# --- QUICK TEST (non-interactive) ---
# add_entry(JOURNAL_FILE, "Started learning about file I/O today!")
# add_entry(JOURNAL_FILE, "Python makes reading files so easy.")
# read_journal(JOURNAL_FILE)
# print("Search results:", search_journal(JOURNAL_FILE, "python"))
# print("Total entries:", count_entries(JOURNAL_FILE))

# run_journal()  # Uncomment to test the interactive menu

---
## Exercise 2: To-Do List Manager (JSON Persistence)
**Objective:** Build a CRUD application that stores data in a JSON file.

Create a to-do list manager where tasks are stored as a list of dictionaries in a `tasks.json` file. Each task has a `title`, a `priority` (high/medium/low), and a `done` status (True/False).

**Tasks:**
1. `load_tasks(filename)` → Load tasks from JSON. Return an empty list if the file is missing or corrupted.
2. `save_tasks(filename, tasks)` → Save the tasks list to JSON with pretty formatting.
3. `add_task(tasks)` → Prompt for title and priority, append a new task dict (done=False).
4. `list_tasks(tasks)` → Display all tasks in a formatted table with their index, status, title, and priority.
5. `mark_done(tasks)` → Ask for a task index and set its `done` field to True.
6. `delete_task(tasks)` → Ask for a task index and remove the task from the list.
7. `run_todo()` → Menu loop: [A]dd, [L]ist, [D]one, [R]emove, [Q]uit. Auto-save after every change.

**Data structure:**
```python
[
    {"title": "Buy groceries", "priority": "high", "done": False},
    {"title": "Read chapter 5", "priority": "medium", "done": True}
]
```

In [34]:
import json

TODO_FILE = "tasks.json"

def load_tasks(filename):
    """Load tasks from a JSON file. Return [] if missing or corrupted."""
    try:
        with open(filename, "r", encoding="utf-8") as file:
            tasks = json.load(file)
    except (FileNotFoundError, json.JSONDecodeError):
        return []
    if not isinstance(tasks, list) or any(not isinstance(task, dict) for task in tasks):
        return []
    return tasks

def save_tasks(filename, tasks):
    """Save tasks list to a JSON file with indent=2."""
    with open(filename, "w", encoding="utf-8") as file:
        json.dump(tasks, file, indent=2)

def add_task(tasks):
    """Prompt for title and priority, then append a new task."""
    title = input("Task title: ").strip()
    if not title:
        print("Task title cannot be empty.")
        return
    priority = input("Priority (high/medium/low): ").strip().lower()
    if priority not in {"high", "medium", "low"}:
        print("Priority must be high, medium, or low.")
        return
    tasks.append({"title": title, "priority": priority, "done": False})
    print("Task added.")

def list_tasks(tasks):
    """Display all tasks in a formatted table."""
    if not tasks:
        print("No tasks found.")
        return
    print(f"{'#':<4} {'Status':<8} {'Title':<30} {'Priority':<8}")
    print("-" * 54)
    for index, task in enumerate(tasks, start=1):
        status = "Done" if task.get("done", False) else "Open"
        print(f"{index:<4} {status:<8} {task.get('title', ''):<30} {task.get('priority', ''):<8}")

def mark_done(tasks):
    """Ask for a task number and mark it as done."""
    if not tasks:
        print("No tasks to mark as done.")
        return
    try:
        index = int(input("Task number to mark done: ")) - 1
    except ValueError:
        print("Enter a valid task number.")
        return
    if not 0 <= index < len(tasks):
        print("Task number is out of range.")
        return
    tasks[index]["done"] = True
    print("Task marked as done.")

def delete_task(tasks):
    """Ask for a task number and remove that task."""
    if not tasks:
        print("No tasks to remove.")
        return
    try:
        index = int(input("Task number to remove: ")) - 1
    except ValueError:
        print("Enter a valid task number.")
        return
    if not 0 <= index < len(tasks):
        print("Task number is out of range.")
        return
    removed = tasks.pop(index)
    print(f"Removed task: {removed.get('title', '')}")

def run_todo():
    """Main menu loop with automatic saving after changes."""
    tasks = load_tasks(TODO_FILE)
    while True:
        print("\n===== TO-DO LIST =====")
        print("[A]dd  [L]ist  [D]one  [R]emove  [Q]uit")
        choice = input("Choose an option: ").strip().lower()
        if choice == "a":
            previous_count = len(tasks)
            add_task(tasks)
            if len(tasks) != previous_count:
                save_tasks(TODO_FILE, tasks)
        elif choice == "l":
            list_tasks(tasks)
        elif choice == "d":
            previous_done = [task.get("done", False) for task in tasks]
            mark_done(tasks)
            if previous_done != [task.get("done", False) for task in tasks]:
                save_tasks(TODO_FILE, tasks)
        elif choice == "r":
            previous_count = len(tasks)
            delete_task(tasks)
            if len(tasks) != previous_count:
                save_tasks(TODO_FILE, tasks)
        elif choice == "q":
            save_tasks(TODO_FILE, tasks)
            print("Tasks saved. Goodbye!")
            break
        else:
            print("Invalid option. Please try again.")


# --- QUICK TEST (non-interactive) ---
# tasks = load_tasks(TODO_FILE)
# tasks.append({"title": "Test task", "priority": "high", "done": False})
# save_tasks(TODO_FILE, tasks)
# tasks = load_tasks(TODO_FILE)
# list_tasks(tasks)

# run_todo()  # Uncomment to test the interactive menu

---
## Exercise 3: CSV Expense Importer
**Objective:** Read, validate, and analyze data from a CSV file.

You receive a CSV file of monthly expenses. Some rows may have missing or invalid data. Write functions to import the data safely, clean it, and produce summary reports.

**First, run the cell below to generate the sample CSV file.** Then write the functions to process it.

**Tasks:**
1. `load_expenses(filename)` → Read the CSV using `csv.DictReader`. Validate each row: skip rows where `amount` is not a valid number or `category` is empty. Return a list of clean dicts (with `amount` converted to float).
2. `total_by_category(expenses)` → Return a dict mapping each category to its total spending.
3. `most_expensive(expenses)` → Return the single most expensive entry.
4. `print_report(expenses)` → Print a formatted summary: total spent, breakdown by category, and the most expensive item.

In [35]:
import csv

# --- RUN THIS CELL FIRST to generate the sample CSV ---
sample_data = """date,description,category,amount
2024-01-05,Grocery store,Food,52.30
2024-01-06,Bus pass,Transport,40.00
2024-01-07,Coffee shop,Food,4.50
2024-01-08,Netflix,Entertainment,15.99
2024-01-09,Gas station,Transport,bad_data
2024-01-10,Restaurant,Food,38.75
2024-01-11,Gym membership,,45.00
2024-01-12,Bookstore,Education,22.50
2024-01-13,Electricity bill,Utilities,89.00
2024-01-14,Movie tickets,Entertainment,24.00
2024-01-15,Taxi,Transport,18.50
2024-01-16,Online course,Education,49.99
2024-01-17,Supermarket,Food,67.20
2024-01-18,Water bill,Utilities,not_a_number
2024-01-19,Concert,Entertainment,75.00
2024-01-20,Pharmacy,Health,12.40"""

with open("expenses.csv", "w", newline="") as f:
    f.write(sample_data)

print("expenses.csv created successfully.")

expenses.csv created successfully.


In [36]:
import csv

def load_expenses(filename):
    """Load and validate expenses from CSV. Skip invalid rows."""
    expenses_list = []

    with open(filename, "r", newline="", encoding="utf-8") as file:
        reader = csv.DictReader(file)

        for row in reader:
            category = (row.get("category") or "").strip()

            try:
                amount = float(row["amount"])
            except (ValueError, TypeError, KeyError):
                continue

            if not category:
                continue

            expenses_list.append({
                "date": row.get("date", ""),
                "desc": row.get("description", ""),
                "category": category,
                "amount": amount,
            })
                

    # TODO: Open file with csv.DictReader
    # TODO: For each row, validate that 'amount' converts to float
    # TODO: Validate that 'category' is not empty
    # TODO: Return list of clean dicts with amount as float
    return expenses_list

def total_by_category(expenses):
    """Return {category: total_amount} dict."""
    # TODO: Accumulation pattern over categories
    category_totals = {}
    for expense in expenses:
        category = expense["category"]
        amount = expense["amount"]
        if category in category_totals:
            category_totals[category] += amount
        else:
            category_totals[category] = amount
    return category_totals

def most_expensive(expenses):
    """Return the highest-amount expense, or None when there are no expenses."""
    if not expenses:
        return None
    return max(expenses, key=lambda x: x["amount"])

def print_report(expenses):
    """Print a formatted expense summary report."""
    if not expenses:
        print("Total spent: $0.00")
        print("No valid expenses to report.")
        return

    total_spent = sum(expense["amount"] for expense in expenses)
    category_breakdown = total_by_category(expenses)
    most_expensive_item = most_expensive(expenses)

    print(f"Total spent: ${total_spent:.2f}")
    print("Breakdown by category:")
    for category, amount in category_breakdown.items():
        print(f"  {category}: ${amount:.2f}")
    print(f"Most expensive item: {most_expensive_item['desc']} (${most_expensive_item['amount']:.2f})")


# --- TEST ---
expenses = load_expenses("expenses.csv")
print_report(expenses)

Total spent: $510.13
Breakdown by category:
  Food: $162.75
  Transport: $58.50
  Entertainment: $114.99
  Education: $72.49
  Utilities: $89.00
  Health: $12.40
Most expensive item: Electricity bill ($89.00)


---
## Exercise 4: File Content Validator
**Objective:** Write robust validation functions for both JSON and CSV files.

You will build a validation toolkit that checks whether files are well-formed and contain the expected structure *before* your application tries to use the data.

**First, run the cell below to generate sample test files** (some valid, some intentionally broken). Then write the validators.

**Tasks:**
1. `validate_json(filename, required_keys)` → Check that the file (a) exists, (b) contains valid JSON, (c) is a list of dicts, and (d) each dict contains all `required_keys`. Return a report dict: `{"valid": int, "invalid": int, "errors": [str]}`.
2. `validate_csv(filename, expected_columns)` → Check that the file (a) exists, (b) has the correct number of columns in the header, and (c) each data row has the correct number of fields. Return a similar report dict.
3. `print_validation_report(report, filename)` → Pretty-print the validation results.

In [37]:
import json

# --- RUN THIS CELL FIRST to generate test files ---

# Good JSON file
good_json = [
    {"name": "Alice", "age": 22, "email": "alice@test.com"},
    {"name": "Bob", "age": 25, "email": "bob@test.com"},
    {"name": "Charlie", "age": 30, "email": "charlie@test.com"},
]
with open("good_data.json", "w") as f:
    json.dump(good_json, f, indent=2)

# Bad JSON file (missing keys, wrong types)
bad_json = [
    {"name": "Alice", "age": 22, "email": "alice@test.com"},
    {"name": "Bob", "age": 25},
    "this is not a dict",
    {"name": "Diana", "age": "not_a_number", "email": "diana@test.com"},
]
with open("bad_data.json", "w") as f:
    json.dump(bad_json, f, indent=2)

# Corrupted JSON file
with open("corrupted.json", "w") as f:
    f.write('{"name": "broken", missing_quotes: true}')

# Good CSV file
with open("good_data.csv", "w", newline="") as f:
    f.write("name,age,email\n")
    f.write("Alice,22,alice@test.com\n")
    f.write("Bob,25,bob@test.com\n")

# Bad CSV file (inconsistent columns)
with open("bad_data.csv", "w", newline="") as f:
    f.write("name,age,email\n")
    f.write("Alice,22,alice@test.com\n")
    f.write("Bob,25\n")
    f.write("Charlie,30,charlie@test.com,extra_field\n")
    f.write("Diana,28,diana@test.com\n")

print("✅ All test files created successfully.")

✅ All test files created successfully.


In [38]:
import json
import csv

def validate_json(filename, required_keys):
    """Validate a JSON file's structure and required keys.
    Return {"valid": int, "invalid": int, "errors": [str]}."""
    # TODO: Try to open and parse the file
    # TODO: Check that root is a list
    # TODO: Check each item is a dict with all required_keys
    # TODO: Collect errors and counts
    report = {"valid": 0, "invalid": 0, "errors": []}
    try:
        with open(filename, "r") as f:
            data = json.load(f)
    except FileNotFoundError:
        report["errors"].append(f"File '{filename}' not found.")
        return report
    except json.JSONDecodeError as e:
        report["errors"].append(f"JSON decode error: {str(e)}")
        return report
    if not isinstance(data, list):
        report["errors"].append("Root element is not a list.")
        return report
    for i, item in enumerate(data):
        if not isinstance(item, dict):
            report["invalid"] += 1
            report["errors"].append(f"Item {i} is not a dict.")
            continue
        missing_keys = [key for key in required_keys if key not in item]
        if missing_keys:
            report["invalid"] += 1
            report["errors"].append(f"Item {i} is missing keys: {missing_keys}.")
        else:
            report["valid"] += 1
    return report

def validate_csv(filename, expected_columns):
    """Validate a CSV file's structure and column count.
    Return {"valid": int, "invalid": int, "errors": [str]}."""
    # TODO: Try to open the file
    # TODO: Check header column count
    # TODO: Check each row's column count
    # TODO: Collect errors and counts
    report = {"valid": 0, "invalid": 0, "errors": []}
    try:    
        with open(filename, "r", newline="", encoding="utf-8") as f:
            reader = csv.reader(f)
            header = next(reader, None)
            if header is None:
                report["errors"].append("CSV file is empty.")
                return report
            if len(header) != expected_columns:
                report["errors"].append(f"Header has {len(header)} columns; expected {expected_columns}.")
                return report
            for i, row in enumerate(reader):
                if len(row) != expected_columns:
                    report["invalid"] += 1
                    report["errors"].append(f"Row {i+2} has {len(row)} columns; expected {expected_columns}.")
                else:
                    report["valid"] += 1
    except FileNotFoundError:
        report["errors"].append(f"File '{filename}' not found.")
    return report

def print_validation_report(report, filename):
    """Pretty-print the validation results."""
    # TODO: Print filename, valid/invalid counts, and error details
    print(f"Validation report for {filename}:")
    print(f"  Valid entries: {report['valid']}")
    print(f"  Invalid entries: {report['invalid']}")
    if report["errors"]:
        print("  Errors:")
        for error in report["errors"]:
            print(f"    - {error}")


# --- TEST ---
print("=" * 50)
report = validate_json("good_data.json", ["name", "age", "email"])
print_validation_report(report, "good_data.json")

print("=" * 50)
report = validate_json("bad_data.json", ["name", "age", "email"])
print_validation_report(report, "bad_data.json")

print("=" * 50)
report = validate_json("corrupted.json", ["name", "age", "email"])
print_validation_report(report, "corrupted.json")

print("=" * 50)
report = validate_csv("good_data.csv", 3)
print_validation_report(report, "good_data.csv")

print("=" * 50)
report = validate_csv("bad_data.csv", 3)
print_validation_report(report, "bad_data.csv")

Validation report for good_data.json:
  Valid entries: 3
  Invalid entries: 0
Validation report for bad_data.json:
  Valid entries: 2
  Invalid entries: 2
  Errors:
    - Item 1 is missing keys: ['email'].
    - Item 2 is not a dict.
Validation report for corrupted.json:
  Valid entries: 0
  Invalid entries: 0
  Errors:
    - JSON decode error: Expecting property name enclosed in double quotes: line 1 column 20 (char 19)
Validation report for good_data.csv:
  Valid entries: 2
  Invalid entries: 0
Validation report for bad_data.csv:
  Valid entries: 2
  Invalid entries: 2
  Errors:
    - Row 3 has 2 columns; expected 3.
    - Row 4 has 4 columns; expected 3.


---
## Exercise 5: Format Converter & Exporter
**Objective:** Convert data between JSON and CSV formats in both directions.

Build a bidirectional converter that can transform data files between JSON and CSV. This is an extremely common real-world task — APIs return JSON, but spreadsheets need CSV, and vice versa.

**Tasks:**
1. `json_to_csv(json_filename, csv_filename)` → Read a JSON file (list of dicts) and write it as a CSV file. Use the keys of the first dict as column headers.
2. `csv_to_json(csv_filename, json_filename)` → Read a CSV file and write it as a JSON file (list of dicts).
3. `export_filtered_csv(json_filename, csv_filename, filter_key, filter_value)` → Read a JSON file, keep only records where `record[filter_key] == filter_value`, and export the filtered data to CSV.
4. Handle edge cases: empty files, missing files, and mismatched structures.

In [39]:
import json
import csv

# --- Setup: create a test JSON file ---
students = [
    {"name": "Alice", "age": 22, "city": "Madrid", "grade": "A"},
    {"name": "Bob", "age": 25, "city": "London", "grade": "B"},
    {"name": "Charlie", "age": 30, "city": "Paris", "grade": "A"},
    {"name": "Diana", "age": 28, "city": "Madrid", "grade": "C"},
    {"name": "Eve", "age": 23, "city": "London", "grade": "A"},
]

with open("students.json", "w") as f:
    json.dump(students, f, indent=2)

print("✅ students.json created.")

✅ students.json created.


In [40]:
import json
import csv

def json_to_csv(json_filename, csv_filename):
    """Convert a JSON file (list of dicts) to a CSV file."""
    try:
        with open(json_filename, "r") as f:
            data = json.load(f)
    except FileNotFoundError:
        print(f"File '{json_filename}' not found.")
        return
    except json.JSONDecodeError as e:
        print(f"JSON decode error: {str(e)}")
        return
    if not isinstance(data, list) or any(not isinstance(row, dict) for row in data):
        print(f"'{json_filename}' must contain a JSON list of objects.")
        return
    if not data:
        with open(csv_filename, "w", newline="", encoding="utf-8"):
            pass
        return

    fieldnames = list(data[0].keys())
    if any(set(row.keys()) != set(fieldnames) for row in data):
        print(f"Records in '{json_filename}' have mismatched fields.")
        return
    try:
        with open(csv_filename, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(data)
    except OSError as e:
        print(f"Could not write '{csv_filename}': {e}")

def csv_to_json(csv_filename, json_filename):
    """Convert a CSV file to a JSON file (list of dicts)."""
    try:
        with open(csv_filename, "r", newline="", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            fieldnames = reader.fieldnames
            data = list(reader)
    except FileNotFoundError:
        print(f"File '{csv_filename}' not found.")
        return
    except csv.Error as e:
        print(f"CSV error: {str(e)}")
        return
    if fieldnames is not None and (
        not all(fieldnames) or len(set(fieldnames)) != len(fieldnames)
    ):
        print(f"'{csv_filename}' has missing or duplicate column names.")
        return
    if any(None in row or any(value is None for value in row.values()) for row in data):
        print(f"'{csv_filename}' contains rows with mismatched columns.")
        return

    try:
        with open(json_filename, "w", encoding="utf-8") as f:
            json.dump(data, f)
    except OSError as e:
        print(f"Could not write '{json_filename}': {e}")
    except TypeError as e:
        print(f"JSON encode error: {e}")

def export_filtered_csv(json_filename, csv_filename, filter_key, filter_value):
    """Export only records where record[filter_key] == filter_value to CSV."""
    try:
        with open(json_filename, "r") as f:
            data = json.load(f)
    except FileNotFoundError:
        print(f"File '{json_filename}' not found.")
        return
    except json.JSONDecodeError as e:
        print(f"JSON decode error: {str(e)}")
        return
    if not isinstance(data, list) or any(not isinstance(row, dict) for row in data):
        print(f"'{json_filename}' must contain a JSON list of objects.")
        return
    if not data:
        with open(csv_filename, "w", newline="", encoding="utf-8"):
            pass
        return

    fieldnames = list(data[0].keys())
    if any(set(row.keys()) != set(fieldnames) for row in data):
        print(f"Records in '{json_filename}' have mismatched fields.")
        return
    filtered_data = [row for row in data if row.get(filter_key) == filter_value]
    try:
        with open(csv_filename, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(filtered_data)
    except OSError as e:
        print(f"Could not write '{csv_filename}': {e}")


# --- TEST ---
# JSON → CSV
json_to_csv("students.json", "students_export.csv")

# CSV → JSON (round-trip test)
csv_to_json("students_export.csv", "students_roundtrip.json")

# Filtered export: only students from Madrid
export_filtered_csv("students.json", "madrid_students.csv", "city", "Madrid")

# Verify round-trip
with open("students_roundtrip.json", "r") as f:
    roundtrip = json.load(f)
print(f"\nRound-trip data ({len(roundtrip)} records):")
for r in roundtrip:
    print(f"  {r}")


Round-trip data (5 records):
  {'name': 'Alice', 'age': '22', 'city': 'Madrid', 'grade': 'A'}
  {'name': 'Bob', 'age': '25', 'city': 'London', 'grade': 'B'}
  {'name': 'Charlie', 'age': '30', 'city': 'Paris', 'grade': 'A'}
  {'name': 'Diana', 'age': '28', 'city': 'Madrid', 'grade': 'C'}
  {'name': 'Eve', 'age': '23', 'city': 'London', 'grade': 'A'}


---
## Exercise 6: Student Records Database (Capstone)
**Objective:** Build a complete file-based CRUD application combining JSON persistence, CSV import/export, and data validation.

Create a student records management system that stores data in a JSON file and supports importing/exporting CSV. This exercise combines **every concept** from this session.

**Required features:**
1. **[A]dd Student** — Prompt for name, age, email, and grade. Validate all inputs (age must be a positive integer, email must contain `@`, grade must be A/B/C/D/F). Store in JSON.
2. **[L]ist Students** — Display all students in a formatted table.
3. **[S]earch** — Find a student by name (case-insensitive partial match).
4. **[U]pdate** — Modify a student's grade or email by name.
5. **[D]elete** — Remove a student by name.
6. **[I]mport CSV** — Load students from a CSV file, validating each row before adding. Skip invalid rows and report what was skipped.
7. **[E]xport CSV** — Export all current students to a CSV file.
8. **[Q]uit** — Exit the program.

**Data structure (JSON):**
```python
[
    {"name": "Alice", "age": 22, "email": "alice@uni.edu", "grade": "A"},
    {"name": "Bob", "age": 25, "email": "bob@uni.edu", "grade": "B"}
]
```

In [43]:
import json
import csv

DB_FILE = "students_db.json"

# --- PERSISTENCE ---

def load_db(filename):
    """Load student records from JSON. Return [] if missing/corrupted."""
    # TODO: try/except for FileNotFoundError and json.JSONDecodeError
    try:
        with open(filename, "r") as f:
            students = json.load(f)
    except FileNotFoundError:
        students = []
    except json.JSONDecodeError:
        students = []
    if not isinstance(students, list) or any(not isinstance(student, dict) for student in students):
        return []
    return students

def save_db(filename, students):
    """Save student records to JSON."""
    # TODO: json.dump with indent=2
    try:
        with open(filename, "w", encoding="utf-8") as f:
            json.dump(students, f, indent=2)
    except OSError as e:
        print(f"Could not write '{filename}': {e}")

# --- VALIDATION ---

def validate_student(name, age, email, grade):
    """Validate student fields. Return (is_valid, error_message)."""
    # TODO: name must be non-empty
    # TODO: age must be a positive integer
    # TODO: email must contain '@'
    # TODO: grade must be one of A, B, C, D, F
    if not name.strip():
        return False, "Name cannot be empty."
    try:
        age_int = int(age)
        if age_int <= 0:
            return False, "Age must be a positive integer."
    except ValueError:
        return False, "Age must be an integer."
    if "@" not in email or not email.strip():
        return False, "Email must contain '@' and cannot be empty."
    if grade not in ["A", "B", "C", "D", "F"]:
        return False, "Grade must be one of A, B, C, D, F."
    return True, ""

# --- CRUD OPERATIONS ---

def add_student(students):
    """Prompt for student info, validate, and add to list."""
    # TODO: Get input, call validate_student, append if valid
    name = input("Enter student name: ").strip()
    age = input("Enter student age: ").strip()
    email = input("Enter student email: ").strip()
    grade = input("Enter student grade: ").strip().upper()

    is_valid, error_message = validate_student(name, age, email, grade)
    if not is_valid:
        print(f"Error adding student: {error_message}")
        return

    student = {"name": name, "age": int(age), "email": email, "grade": grade}
    students.append(student)
    print("Student added successfully.")

def list_students(students):
    """Display all students in a formatted table."""
    # TODO: Handle empty list
    # TODO: Print formatted table with headers
    if not students:
        print("No students in the database.")
        return

    print("Student Records:")
    print("-" * 60)
    print(f"{'Name':<20} {'Age':<10} {'Email':<25} {'Grade':<10}")
    print("-" * 60)
    for student in students:
        print(f"{student['name']:<20} {student['age']:<10} {student['email']:<25} {student['grade']:<10}")

def search_student(students):
    """Search by name (case-insensitive partial match)."""
    # TODO: Get search term
    # TODO: Filter students whose name contains the term
    term = input("Enter name to search: ").strip().lower()
    matches = [s for s in students if term in s["name"].lower()]
    if not matches:
        print("No matching students found.")
        return
    print(f"Found {len(matches)} matching student(s):")
    for student in matches:
        print(f"  Name: {student['name']}, Age: {student['age']}, Email: {student['email']}, Grade: {student['grade']}")

def update_student(students):
    """Update a student's grade or email by name."""
    # TODO: Find student by name
    # TODO: Ask which field to update
    # TODO: Validate new value before applying
    name = input("Enter the name of the student to update: ").strip()
    matches = [s for s in students if s["name"].lower() == name.lower()]
    if not matches:
        print("No student found with that name.")
        return
    student = matches[0]
    print(f"Current info: Name: {student['name']}, Age: {student['age']}, Email: {student['email']}, Grade: {student['grade']}")
    field = input("Update [G]rade or [E]mail? ").strip().lower()
    if field == "g":
        new_grade = input("New grade (A/B/C/D/F): ").strip().upper()
        is_valid, error_message = validate_student(
            student["name"], student["age"], student["email"], new_grade
        )
        if not is_valid:
            print(f"Could not update student: {error_message}")
            return
        student["grade"] = new_grade
    elif field == "e":
        new_email = input("New email: ").strip()
        is_valid, error_message = validate_student(
            student["name"], student["age"], new_email, student["grade"]
        )
        if not is_valid:
            print(f"Could not update student: {error_message}")
            return
        student["email"] = new_email
    else:
        print("Choose G to update the grade or E to update the email.")
        return
    print("Student updated successfully.")

def delete_student(students):
    """Remove a student by name."""
    name = input("Enter the name of the student to delete: ").strip()
    matches = [student for student in students if student["name"].lower() == name.lower()]
    if not matches:
        print("No student found with that name.")
        return
    student = matches[0]
    confirmation = input(f"Delete {student['name']}? (y/n): ").strip().lower()
    if confirmation == "y":
        students.remove(student)
        print("Student deleted.")
    else:
        print("Deletion cancelled.")

# --- IMPORT / EXPORT ---

def import_csv(students, csv_filename):
    """Import students from CSV, validating each row."""
    required_columns = {"name", "age", "email", "grade"}
    imported = 0
    skipped = 0
    try:
        with open(csv_filename, "r", newline="", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            if reader.fieldnames is None or not required_columns.issubset(reader.fieldnames):
                print(f"CSV must contain these columns: {', '.join(sorted(required_columns))}.")
                return
            for row_number, row in enumerate(reader, start=2):
                if None in row:
                    skipped += 1
                    print(f"Skipping row {row_number}: too many columns.")
                    continue
                name = (row.get("name") or "").strip()
                age = (row.get("age") or "").strip()
                email = (row.get("email") or "").strip()
                grade = (row.get("grade") or "").strip().upper()
                is_valid, error_message = validate_student(name, age, email, grade)
                if not is_valid:
                    skipped += 1
                    print(f"Skipping row {row_number}: {error_message}")
                    continue
                students.append({"name": name, "age": int(age), "email": email, "grade": grade})
                imported += 1
    except FileNotFoundError:
        print(f"File '{csv_filename}' not found.")
        return
    except csv.Error as e:
        print(f"CSV error in '{csv_filename}': {e}")
        return
    print(f"Imported {imported} student(s); skipped {skipped} invalid row(s).")

def export_csv(students, csv_filename):
    """Export all students to a CSV file."""
    fieldnames = ["name", "age", "email", "grade"]
    try:
        with open(csv_filename, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(
                {field: student.get(field, "") for field in fieldnames}
                for student in students
            )
    except OSError as e:
        print(f"Could not write '{csv_filename}': {e}")

# --- MAIN MENU ---

def run_student_db():
    """Main menu loop."""
    students = load_db(DB_FILE)
    print(f"Loaded {len(students)} student(s) from database.")

    while True:
        print("\n===== STUDENT RECORDS DATABASE =====")
        print("[A]dd  [L]ist  [S]earch  [U]pdate  [D]elete")
        print("[I]mport CSV  [E]xport CSV  [Q]uit")
        choice = input("> ").strip().lower()

        if choice == "a":
            add_student(students)
            save_db(DB_FILE, students)
        elif choice == "l":
            list_students(students)
        elif choice == "s":
            search_student(students)
        elif choice == "u":
            update_student(students)
            save_db(DB_FILE, students)
        elif choice == "d":
            delete_student(students)
            save_db(DB_FILE, students)
        elif choice == "i":
            csv_file = input("CSV filename to import: ").strip()
            import_csv(students, csv_file)
            save_db(DB_FILE, students)
        elif choice == "e":
            csv_file = input("CSV filename to export to: ").strip()
            export_csv(students, csv_file)
        elif choice == "q":
            save_db(DB_FILE, students)
            print("Database saved. Goodbye!")
            break
        else:
            print("Invalid option. Please try again.")

# run_student_db()  # Uncomment to test

In [44]:
# --- Generate a sample CSV for testing the Import feature ---

sample_import = """name,age,email,grade
Frank,21,frank@uni.edu,B
Grace,23,grace@uni.edu,A
Hank,not_a_number,hank@uni.edu,C
Ivy,20,ivy_no_at_sign,A
Jack,22,jack@uni.edu,Z
Karen,24,karen@uni.edu,B"""

with open("import_students.csv", "w", newline="") as f:
    f.write(sample_import)

print("import_students.csv created (3 valid, 3 invalid rows).")

import_students.csv created (3 valid, 3 invalid rows).
